# Download via HTTP API

This notebook demonstrates how to interact with the `media-downloader` server from any HTTP client.

Two patterns are covered:

1. **`POST /download`** - synchronous: blocks until the download is complete and returns the result directly.
2. **`POST /jobs` + `GET /jobs/{id}`** - async queue: submit a job and poll for its status.

**Prerequisite:** the server must be running:

```
uvicorn media_downloader.webapp.app:app --reload
```


## Imports


In [ ]:
import time

import httpx
from rich import print as rprint


## Configuration


In [ ]:
BASE_URL = "http://localhost:8000"

# Change this URL to whatever you want to download
# URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
# URL = "https://www.instagram.com/reel/DVvd_q1CsWE/"
URL = "https://www.instagram.com/reel/DWCkc8pj1mu/"


## Health check


In [ ]:
resp = httpx.get(f"{BASE_URL}/health")
resp.raise_for_status()
rprint(resp.json())


## Synchronous download - `POST /download`

Blocks until the download (and transcription, if configured) is complete.
Returns source info, files, caption, and optional transcript.


In [ ]:
resp = httpx.post(
    f"{BASE_URL}/download",
    json={"url": URL},
    timeout=120.0,  # downloads can take a while
)
resp.raise_for_status()
result = resp.json()
rprint(result)


In [ ]:
# Inspect the response fields
print("Source:     ", result["source"])
print("Source ID:  ", result["source_id"])
print("Caption:    ", result.get("caption", "")[:120])
print("Transcript: ", result.get("transcript"))
print("Language:   ", result.get("language"))
print("Files:")
for f in result.get("files", []):
    print("  -", f)


## Async job queue - `POST /jobs` + `GET /jobs/{id}`

Submit a job to the background worker and poll for completion.
Useful for long downloads where you don't want to block the HTTP connection.


In [ ]:
# Submit the job (returns immediately with status=pending)
resp = httpx.post(
    f"{BASE_URL}/jobs",
    json={"url": URL},
)
resp.raise_for_status()
job = resp.json()
job_id = job["id"]
print(f"Job submitted: {job_id}  status={job['status']}")


In [ ]:
# Poll until the job is done
POLL_INTERVAL = 5  # seconds between checks

while True:
    resp = httpx.get(f"{BASE_URL}/jobs/{job_id}")
    resp.raise_for_status()
    job = resp.json()
    status = job["status"]
    print(f"  status={status}")

    if status == "completed":
        rprint(job)
        break

    if status == "failed":
        print(f"  ERROR: {job['error']}")
        break

    time.sleep(POLL_INTERVAL)


In [ ]:
# Inspect the completed job
if job["status"] == "completed":
    print("Source:     ", job["source"])
    print("Source ID:  ", job["source_id"])
    print("Transcript: ", job.get("transcript"))
    print("Language:   ", job.get("language"))
    print("Files:")
    for mf in job.get("media_files", []):
        print(f"  [{mf['role']}]  {mf['file_path']}  (on_disk={mf['media_on_disk']})")
